# CIFAR-10 — Optimised (GPU / TPU auto-detect)

Rewrite of the original notebook. Key changes:

| Problem in original | Fix here |
|---|---|
| TPU runtime selected but TF never used it → CPU training at ~161 ms/step | `get_strategy()` auto-detects TPU/GPU/CPU and builds the model inside the correct scope |
| Raw NumPy arrays fed to `model.fit` | `tf.data` pipeline: cache → shuffle → batch → prefetch(AUTOTUNE) |
| float32 images held in RAM (600 MB) | uint8 in memory, `Rescaling` on-device |
| Per-image Python loop to decode 60k PNGs | `ds.with_format("numpy")` — vectorised |
| No augmentation → plateaus ~72% | Random flip/translate/zoom layers, on-device |
| Fixed LR, 15 epochs | Cosine decay + warmup, AdamW, early stopping |
| Batch size 64 | Scaled per replica (512+ on TPU) |
| Full precision | Mixed precision (`float16` GPU / `bfloat16` TPU) |

**Runtime:** `T4 GPU` is the pragmatic choice for a model this small. `v5e-1 TPU` also works.


## 0. Setup

In [ ]:
# Colab has these already; uncomment on a bare VM.
# !pip install -q tensorflow datasets

import os, time, numpy as np, matplotlib.pyplot as plt
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers

print("TensorFlow:", tf.__version__)

## 1. Device strategy — this is the part that was missing

In [ ]:
def get_strategy():
    """Return (strategy, label). Order of preference: TPU -> GPU -> CPU."""
    try:
        resolver = tf.distribute.cluster_resolver.TPUClusterResolver("")
        tf.config.experimental_connect_to_cluster(resolver)
        tf.tpu.experimental.initialize_tpu_system(resolver)
        return tf.distribute.TPUStrategy(resolver), "TPU"
    except Exception:
        pass

    gpus = tf.config.list_physical_devices("GPU")
    if gpus:
        for g in gpus:                      # avoid grabbing all VRAM up front
            tf.config.experimental.set_memory_growth(g, True)
        if len(gpus) > 1:
            return tf.distribute.MirroredStrategy(), "GPU x%d" % len(gpus)
        return tf.distribute.get_strategy(), "GPU"

    return tf.distribute.get_strategy(), "CPU"


strategy, DEVICE = get_strategy()
N_REPLICAS = strategy.num_replicas_in_sync

# Mixed precision: bfloat16 is native on TPU, float16 uses tensor cores on T4/A100.
if DEVICE.startswith("TPU"):
    keras.mixed_precision.set_global_policy("mixed_bfloat16")
elif DEVICE.startswith("GPU"):
    keras.mixed_precision.set_global_policy("mixed_float16")

print(f"Device: {DEVICE}  |  replicas: {N_REPLICAS}")
print("Precision policy:", keras.mixed_precision.global_policy().name)

## 2. Load data — vectorised

`ds.with_format("numpy")` decodes the whole split in one shot instead of a
`[np.array(im) for im in split["img"]]` Python loop. Images stay **uint8**:
60k × 32 × 32 × 3 = 184 MB instead of 737 MB as float32.

In [ ]:
from datasets import load_dataset

ds = load_dataset("uoft-cs/cifar10")

class_names = ["airplane", "automobile", "bird", "cat", "deer",
               "dog", "frog", "horse", "ship", "truck"]

def to_numpy(split):
    d = split.with_format("numpy")[:]          # [:] materialises the lazy Column
    imgs   = np.asarray(d["img"],   dtype="uint8")
    labels = np.asarray(d["label"], dtype="int32")
    return imgs, labels

x_train_full, y_train_full = to_numpy(ds["train"])
x_test,       y_test       = to_numpy(ds["test"])

# Hold out a validation split once, deterministically.
rng = np.random.default_rng(42)
idx = rng.permutation(len(x_train_full))
n_val = 5000
val_idx, tr_idx = idx[:n_val], idx[n_val:]

x_val,   y_val   = x_train_full[val_idx], y_train_full[val_idx]
x_train, y_train = x_train_full[tr_idx],  y_train_full[tr_idx]

print(f"Train {x_train.shape} | Val {x_val.shape} | Test {x_test.shape}")
print(f"Memory: {x_train_full.nbytes / 1e6:.0f} MB (uint8)")

## 3. `tf.data` pipeline

`cache()` keeps the decoded tensors in RAM so epoch 2+ skips all host work.
`prefetch(AUTOTUNE)` overlaps host preprocessing with device compute — this is
what stops the accelerator from idling between steps.

`drop_remainder=True` is **required** on TPU (static shapes) and harmless on GPU.

In [ ]:
AUTOTUNE = tf.data.AUTOTUNE

# Per-replica batch of 256; TPU wants big batches to stay fed.
PER_REPLICA_BATCH = 256 if DEVICE.startswith("TPU") else 128
BATCH_SIZE = PER_REPLICA_BATCH * N_REPLICAS

def make_ds(x, y, training: bool):
    d = tf.data.Dataset.from_tensor_slices((x, y))
    d = d.cache()
    if training:
        d = d.shuffle(len(x), seed=42, reshuffle_each_iteration=True)
    d = d.batch(BATCH_SIZE, drop_remainder=training)
    return d.prefetch(AUTOTUNE)

train_ds = make_ds(x_train, y_train, training=True)
val_ds   = make_ds(x_val,   y_val,   training=False)
test_ds  = make_ds(x_test,  y_test,  training=False)

steps_per_epoch = len(x_train) // BATCH_SIZE
print(f"Global batch: {BATCH_SIZE} | steps/epoch: {steps_per_epoch}")

## 4. Model

Augmentation and rescaling live **inside** the model, so they run on the
accelerator rather than the CPU. Augmentation layers are inert at inference.

The final `Dense` is forced to `float32` — softmax in float16 overflows.

In [ ]:
def build_model():
    reg = keras.regularizers.l2(1e-4)

    def conv_block(x, filters, blocks=2, drop=0.2):
        for _ in range(blocks):
            x = layers.Conv2D(filters, 3, padding="same",
                              use_bias=False, kernel_regularizer=reg)(x)
            x = layers.BatchNormalization()(x)
            x = layers.Activation("relu")(x)
        x = layers.MaxPooling2D()(x)
        return layers.Dropout(drop)(x)

    inputs = keras.Input(shape=(32, 32, 3), dtype="uint8")

    x = layers.Rescaling(1.0 / 255)(inputs)   # was: (tf.cast(inputs, "float32"))
    x = layers.RandomFlip("horizontal")(x)
    x = layers.RandomTranslation(0.1, 0.1, fill_mode="reflect")(x)
    x = layers.RandomZoom(0.1, fill_mode="reflect")(x)

    x = conv_block(x,  64, blocks=2, drop=0.20)   # 32 -> 16
    x = conv_block(x, 128, blocks=2, drop=0.30)   # 16 -> 8
    x = conv_block(x, 256, blocks=3, drop=0.40)   # 8  -> 4

    x = layers.GlobalAveragePooling2D()(x)        # replaces Flatten + Dense(128)
    x = layers.Dropout(0.3)(x)
    outputs = layers.Dense(10, dtype="float32")(x)   # logits, float32

    return keras.Model(inputs, outputs, name="cifar10_cnn")


EPOCHS = 60

with strategy.scope():
    model = build_model()

    lr_schedule = keras.optimizers.schedules.CosineDecay(
        initial_learning_rate=0.0,
        warmup_target=1e-3 * (BATCH_SIZE / 128),   # linear LR scaling
        warmup_steps=steps_per_epoch * 3,
        decay_steps=steps_per_epoch * (EPOCHS - 3),
        alpha=0.01,
    )

    model.compile(
        optimizer=keras.optimizers.AdamW(learning_rate=lr_schedule, weight_decay=1e-4),
        loss=keras.losses.SparseCategoricalCrossentropy(from_logits=True),
        metrics=[keras.metrics.SparseCategoricalAccuracy(name="accuracy")],
        # Fuses many steps into one device call. Critical on TPU, helps on GPU.
        steps_per_execution=32 if DEVICE.startswith("TPU") else 8,
    )

model.summary()

## 5. Train

In [ ]:
callbacks = [
    keras.callbacks.EarlyStopping(
        monitor="val_accuracy", patience=12,
        restore_best_weights=True, verbose=1),
    keras.callbacks.ModelCheckpoint(
        "best_cifar10.keras", monitor="val_accuracy",
        save_best_only=True, verbose=0),
]

t0 = time.time()
history = model.fit(
    train_ds,
    epochs=EPOCHS,
    validation_data=val_ds,
    callbacks=callbacks,
    verbose=1,
)
elapsed = time.time() - t0
print(f"\nTrained on {DEVICE} in {elapsed/60:.1f} min "
      f"({elapsed/len(history.history['loss']):.1f} s/epoch)")

## 6. Result

In [ ]:
test_loss, test_acc = model.evaluate(test_ds, verbose=0)
print(f"Test images:   {len(x_test):,}")
print(f"Test loss:     {test_loss:.4f}")
print(f"Test accuracy: {test_acc * 100:.2f}%")

In [ ]:
plt.figure(figsize=(11, 4))

plt.subplot(1, 2, 1)
plt.plot(np.array(history.history["accuracy"]) * 100, label="train")
plt.plot(np.array(history.history["val_accuracy"]) * 100, label="val")
plt.xlabel("Epoch"); plt.ylabel("Accuracy (%)"); plt.legend(); plt.grid(alpha=.3)
plt.title("Accuracy")

plt.subplot(1, 2, 2)
plt.plot(history.history["loss"], label="train")
plt.plot(history.history["val_loss"], label="val")
plt.xlabel("Epoch"); plt.ylabel("Loss"); plt.legend(); plt.grid(alpha=.3)
plt.title("Loss")

plt.tight_layout(); plt.show()

## 7. Predictions, per-class accuracy, confusion matrix

In [ ]:
logits = model.predict(test_ds, verbose=0)
probs  = tf.nn.softmax(logits).numpy()
preds  = probs.argmax(axis=1)

y_true = y_test[:len(preds)]   # drop_remainder=False on test, so this is a no-op

plt.figure(figsize=(12, 6))
for i in range(12):
    plt.subplot(3, 4, i + 1)
    plt.imshow(x_test[i])
    conf = probs[i][preds[i]] * 100
    ok = preds[i] == y_true[i]
    plt.title(f"{class_names[preds[i]]} ({conf:.1f}%)\ntrue: {class_names[y_true[i]]}",
              fontsize=8, color="green" if ok else "red")
    plt.axis("off")
plt.tight_layout(); plt.show()

In [ ]:
from sklearn.metrics import classification_report, confusion_matrix
import seaborn as sns

print(classification_report(y_true, preds, target_names=class_names, digits=3))

cm = confusion_matrix(y_true, preds)
plt.figure(figsize=(8, 6))
sns.heatmap(cm, annot=True, fmt="d", cmap="Blues",
            xticklabels=class_names, yticklabels=class_names)
plt.xlabel("Predicted"); plt.ylabel("True"); plt.title("Confusion Matrix")
plt.show()

print("Per-class accuracy:")
for i, name in enumerate(class_names):
    print(f"  {name:12s} {cm[i, i] / cm[i].sum() * 100:5.1f}%")

## 8. Single-image prediction

In [ ]:
idx = 42
img = x_test[idx]

p = tf.nn.softmax(model.predict(img[np.newaxis, ...], verbose=0))[0].numpy()
top3 = p.argsort()[-3:][::-1]

plt.imshow(img); plt.axis("off")
plt.title(f"Predicted: {class_names[top3[0]]}")
plt.show()

print("Top 3:")
for j in top3:
    print(f"  {class_names[j]:12s} {p[j] * 100:5.2f}%")
print(f"\nActual: {class_names[y_test[idx]]}")